In [2]:
import pandas as pd
from IPython.display import Markdown, display

In [3]:
def load_excel_file(sheet_name):
  LOCAL_EXCEL_FILE_PATH = "../data/raw/covid_19_dataset.xlsx"
  REMOTE_EXCEL_FILE_PATH = "https://raw.githubusercontent.com/chetanchandel31/da-covid19/main/data/raw/covid_19_dataset.xlsx"
  try:
    return pd.read_excel(LOCAL_EXCEL_FILE_PATH, sheet_name=sheet_name) 
  except FileNotFoundError:
    return pd.read_excel(REMOTE_EXCEL_FILE_PATH, sheet_name=sheet_name, engine="openpyxl")
  except Exception as e:
    print("unexpected error while loading data", e)
    raise

df_deaths = load_excel_file("covid_19_deaths_v1")
df_confirmed = load_excel_file("covid_19_confirmed_v1")
df_recovered = load_excel_file("covid_19_recovered_v1")

In [4]:
for name, df in [("Deaths", df_deaths), ("Confirmed", df_confirmed), ("Recovered", df_recovered)]:
    display(Markdown(f"## **{name}**"))
    display(Markdown(f"**Shape:** {df.shape}"))

    # ~494 columns, too much noise in output
    # print("Columns:", end=" ")
    # print(*df.columns, sep=", ")

    non_date_cols = ["Province/State", "Country/Region", "Lat", "Long"]
    date_cols = [c for c in df.columns if c not in non_date_cols]
    display(Markdown(f"**Non date columns**: {len(non_date_cols)} ({', '.join(non_date_cols)})"))
    display(Markdown(f"**Date columns:** {len(date_cols)} (from {date_cols[0]} to {date_cols[-1]})"))
    display(Markdown(f"*Subset of date column headers*: {date_cols[0:15]}"))

    df_dtypes = df.dtypes
    display(Markdown("**Column wise data types**:"))
    display(df_dtypes)

    date_col_dtypes = set()
    for idx, val in df_dtypes.items():
        if idx not in non_date_cols:
            date_col_dtypes.add(val)

    display(Markdown(f"**Unique data types for date-columns' values:** {list(date_col_dtypes)}"))
    print()


## **Deaths**

**Shape:** (276, 498)

**Non date columns**: 4 (Province/State, Country/Region, Lat, Long)

**Date columns:** 494 (from 1/22/20 to 5/29/21)

*Subset of date column headers*: ['1/22/20', '1/23/20', '1/24/20', '1/25/20', '1/26/20', '1/27/20', '1/28/20', '1/29/20', '1/30/20', '1/31/20', datetime.datetime(2020, 1, 2, 0, 0), datetime.datetime(2020, 2, 2, 0, 0), datetime.datetime(2020, 3, 2, 0, 0), datetime.datetime(2020, 4, 2, 0, 0), datetime.datetime(2020, 5, 2, 0, 0)]

...

**Date columns:** 494 (from 1/22/20 to 5/29/21)

*Subset of date column headers*: ['1/22/20', '1/23/20', '1/24/20', '1/25/20', '1/26/20', '1/27/20', '1/28/20', '1/29/20', '1/30/20', '1/31/20', datetime.datetime(2020, 1, 2, 0, 0), datetime.datetime(2020, 2, 2, 0, 0), datetime.datetime(2020, 3, 2, 0, 0), datetime.datetime(2020, 4, 2, 0, 0), datetime.datetime(2020, 5, 2, 0, 0)]

**Column wise data types**:

Province/State        str
Country/Region        str
Lat               float64
Long              float64
1/22/20             int64
                   ...   
5/25/21             int64
5/26/21             int64
5/27/21             int64
5/28/21             int64
5/29/21             int64
Length: 498, dtype: object

**Unique data types for date-columns' values:** [dtype('int64'), dtype('float64')]

### Structure of raw dataset

- **Shape of data:** Each sheet has 498 columns, suggesting cases recorded over same dates in each sheet. However, the sheet with recovered cases has fewer rows(261) than the other 2 sheets(containing 276 rows each), suggesting recovered cases recorded over comparatively fewer locations.

- **Datatypes of non-date column values:** Location related columns(`Province/State`, `Country/Region`) are correctly typed as `str` and the coordinated(`Lat` and `Long` columns) are `float64`.

- **Datatypes under date-column values:** Values present in date-columns under "confirmed cases" are correctly typed as `int64` confirming zero `NaN` values there. In the other 2 sheets, individual date-columns are typed as either `int64` or `float64`, some columns are one, some are the other, indicating `NaN`s are present in at least some of those columns (`NaN` forces a column to upcast to `float64`), since "number of cases" generally can't be a decimal number.

- **Discrepancy in column header datatype:** Date *column headers*, however, are inconsistently typed - a mix of `str`
  (e.g. `'1/22/20'`) and `datetime.datetime` objects (e.g. `datetime(2020, 1, 2)`), likely due to inconsistent cell formatting in the original Excel file. This will need to be normalized to a single consistent type before melting
  to long format for time-series analysis.